In [1]:
import ctfr
import numpy as np

In [32]:
NUM_ITER = 5

In [33]:
AUDIO_PATH = "audio/guitarset_sample.wav"
SR = 22050
OFFSET = 5.0
DURATION = 4.0

N_FFT = 2048
HOP_LENGTH = 256
WIN_LENGTHS = [512, 1024, 2048]

In [34]:
signal, _ = ctfr.load(AUDIO_PATH, sr=SR, offset=OFFSET, duration=DURATION)
specs = [ctfr.stft_spec(signal, win_length=l, n_fft=N_FFT, hop_length=HOP_LENGTH) for l in WIN_LENGTHS]
print(f"Spectrogram shape: {specs[0].shape}")

Spectrogram shape: (1025, 345)


In [35]:
def time_method(method, **kwargs):
    total_time = 0.0
    for _ in range(NUM_ITER):
        swgm_spec, elapsed_time = ctfr.ctfr_from_specs(specs, method=method, **kwargs)
        total_time += elapsed_time
    average_time = total_time / NUM_ITER
    return swgm_spec, average_time

def evaluate_baseline_and_ctfr(method, has_baseline=True):
    if has_baseline:
        cspec_base, average_time_base = time_method(method=f"baseline_{method}")
    cspec_ctfr, average_time_ctfr = time_method(method=method)

    print(f"==== {ctfr.get_method_name(method)} ====", end="\n\n")
    if has_baseline:
        print(f"Baseline: {average_time_base:0.3f} s -- {100*average_time_base/DURATION:0.2f}% real-time")
    print(f"ctfr: {average_time_ctfr:0.3f} s -- {100*average_time_ctfr/DURATION:0.2f}% real-time")
    if has_baseline:
        assert np.allclose(cspec_base, cspec_ctfr)

In [36]:
evaluate_baseline_and_ctfr("swgm")

==== Sample-weighted geometric mean (SWGM) ====

Baseline: 0.069 s -- 1.72% real-time
ctfr: 0.031 s -- 0.78% real-time


In [37]:
evaluate_baseline_and_ctfr("fls")

==== Fast local sparsity (FLS) ====

Baseline: 0.083 s -- 2.07% real-time
ctfr: 0.073 s -- 1.82% real-time


In [28]:
evaluate_baseline_and_ctfr("lt")

==== Lukin-Todd (LT) ====

Baseline: 6.111 s -- 152.77% real-time
ctfr: 0.815 s -- 20.37% real-time
